# Using MCP Tools with Pydantic

In this notebook, we will explore how create a reasoning action agent using tools exposed by a MCP server with Pydantic.

## Install Dependencies

In [ ]:
!pip install pydantic_ai mcp_server_time

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

## Launch vLLM in the Background

Execute the cell below to create the vllm_serve.sh

In [ ]:
vllm_file = f"""#!/bin/bash

VLLM_USE_TRITON_FLASH_ATTN=0 \\
vllm serve Qwen/Qwen3-30B-A3B \\
    --served-model-name Qwen3-30B-A3B \\
    --api-key abc-123 \\
    --port 8000 \\
    --enable-auto-tool-choice \\
    --tool-call-parser hermes \\
    --trust-remote-code 2>&1 | tee vllm_serve.log
"""

with open('vllm_serve.sh', 'w', encoding='utf-8') as f:
    f.write(vllm_file)

Open a new terminal to execute the `vllm_serve.sh` file. This will serve an LLM locally.

In Jupyter, open a new terminal. `File > New > Terminal`, copy the content below and execute it.

```sh
bash vllm_serve.sh
```

Now, the LLM will be ready to be used once you see `Application startup complete.`

## Create Model Object

Use `OpenAIChatModel` to connect to the Ollama local endpoint. 

In [ ]:
provider = OllamaProvider(
    base_url='http://localhost:8000/v1',
    api_key='abc-123',
)

agent_model = OpenAIChatModel(model_name='Qwen3-30B-A3B', provider=provider)

## Create a Pydantic Agent

Let create the Agent objet

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

## Create Async Agent

Let's start by creating a simple agent with no tools. Note that we are initializing a MCP server session, however, we have not launch any MCP server.

In [ ]:
import asyncio
async def run_agent(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

Let's ask a simple question

In [ ]:
reponse = await run_agent("What is the capital of France?")
print(reponse)

Now, let's ask a question that the model cannot answer without access to tools.

In [ ]:
reponse = await run_agent("What time is it?")
print(reponse)

## Run a MCP server

Now, we can run a MCP server that enables us to get information about the current date and time. 

In [ ]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=Europe/Dublin",
    ],
)

Let's update the Agent object to include the tools exposed by this MCP server. Note that we are manually providing the system prompt.

In [ ]:
agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Because the run_agent makes a reference to the agent object we just updated, we can query the agent and it will first reflect and then act.

In [ ]:
await run_agent("What's the date and time today in Dublin?")

In [ ]:
await run_agent("What's the time Tokio?")

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT